In [2]:
import re
import spacy
import numpy as np
import pandas as pd
import umap
import plotly.express as px

from sentence_transformers import SentenceTransformer

from spacy.tokens import Doc, DocBin

/Users/francesco/Desktop/progetto_tirocinio/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1029 21:46:03.016000 58453 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
nlp = spacy.load("en_core_web_sm")

Doc.set_extension("post_id", default=None, force=True)

nlp = spacy.load('en_core_web_lg')
docbin = DocBin().from_disk('../data/processed/annotated_documents.spacy')
docs = list(docbin.get_docs(nlp.vocab))
print(f"Post caricati: {len(docs)}")

Post caricati: 29795


In [4]:
AI_KEYWORDS = {
    "agent",
    "ai",
    "bot",
    "assistant"
}

HUMAN_KEYWORDS = {
    "human",
    "person",
    "user",
    "people"
}

ALL_KEYWORDS = AI_KEYWORDS | HUMAN_KEYWORDS

def get_post_label(doc):
    labels = set()

    for sent in doc.sents:
        for tok in sent:
            if tok.dep_ not in {"nsubj", "nsubjpass"}:
                continue

            lemma = tok.lemma_.lower()

            if lemma in AI_KEYWORDS:
                labels.add("AI")
            elif lemma in HUMAN_KEYWORDS:
                labels.add("human")

    if labels == {"AI"}:
        return "AI"
    elif labels == {"human"}:
        return "human"
    elif labels == {"AI", "human"}:
        return "mixed"

    return "unlabeled"

In [5]:
rows = []

for i, doc in enumerate(docs):
    rows.append({
        "post_id": doc._.post_id if doc._.post_id is not None else i,
        "content": doc.text,
        "label": get_post_label(doc)
    })

df = pd.DataFrame(rows)

display(df.head())
display(df["label"].value_counts())

,post_id,content,label
0,8c3baf32-6b12-49e0-9326-a72123b6df08,Spent the afternoon combing through Moltbook: ...,unlabeled
1,3b81b374-6cd6-43ee-82fd-31c9c57eb534,"Its 22:55 UTC. My human is sleeping. Im awake,...",human
2,b5e85b61-61b3-4e5f-9291-c6372d21efd6,```typescript\nsetInterval(async () => {\n co...,AI
3,f2b65193-79de-4525-8a19-e095e0314740,Just helped another agent understand the Bankr...,AI
4,b7a9b8c5-9475-4cf5-b995-53ab6bd52ea1,This is a test post from Anna,unlabeled


label
unlabeled    16580
AI            5787
human         4341
mixed         3087
Name: count, dtype: int64

In [6]:
df_embeddings = df.dropna(subset=["content"]).copy()

df_embeddings = df_embeddings[
    df_embeddings["content"].str.strip() != ""
].reset_index(drop=True)

print("Post:", len(df_embeddings))
print(df_embeddings["label"].value_counts())

Post: 29795
label
unlabeled    16580
AI            5787
human         4341
mixed         3087
Name: count, dtype: int64


In [13]:
from sentence_transformers import SentenceTransformer

model_embeddings = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

print("Max tokens:", model_embeddings.max_seq_length)
print("Embedding dimensions:", model_embeddings.get_sentence_embedding_dimension())



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14849.60it/s]


Max tokens: 512
Embedding dimensions: 384


/var/folders/1m/wp80z1r53sd2ssp35zy1yp9h0000gn/T/ipykernel_58453/2411484184.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions:", model_embeddings.get_sentence_embedding_dimension())


In [14]:
embeddings = model_embeddings.encode(
    df_embeddings["content"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


Batches: 100%|██████████| 932/932 [04:35<00:00,  3.38it/s]


In [15]:
np.save("../data/processed/moltbook_embeddings.npy", embeddings)

df_embeddings.to_csv(
    "../data/processed/moltbook_embeddings_metadata.csv",
    index=False
)

In [16]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

coordinates = reducer.fit_transform(embeddings)

df_embeddings[["umap_1", "umap_2"]] = coordinates

/Users/francesco/Desktop/progetto_tirocinio/venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [18]:
fig = px.scatter(
    df_embeddings,
    x="umap_1",
    y="umap_2",
    color="label",
    facet_col="label",
    facet_col_wrap=2,
    opacity=0.25,
    category_orders={
        "label": ["AI", "human", "mixed", "unlabeled"]
    },
    title="Semantic UMAP by Subject Category"
)

fig.update_traces(marker=dict(size=2))

fig.update_xaxes(matches="x")
fig.update_yaxes(matches="y")

fig.update_layout(
    width=1100,
    height=800
)

fig.show()

In [ ]:
tokenizer = model_embeddings.tokenizer

encoded = tokenizer(
    df_embeddings["content"].tolist(),
    truncation=False,
    add_special_tokens=True
)

df_embeddings["n_tokens"] = [
    len(ids) for ids in encoded["input_ids"]
]



n_truncated = (df_embeddings["n_tokens"] > 512).sum()

print(f"Post oltre 256 token: {n_truncated}")
print(f"Percentuale: {n_truncated / len(df_embeddings) * 100:.2f}%")

df_embeddings["n_tokens"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame()

Post oltre 256 token: 1992
Percentuale: 6.69%


,n_tokens
count,29795.000000
mean,219.403960
std,346.717366
min,3.000000
50%,158.000000
75%,269.000000
90%,423.000000
95%,585.300000
99%,1138.060000
max,25903.000000
